In [9]:
# Load data (if not already loaded)
# Uncomment if needed:
import sys
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLoad").getOrCreate()
df_2018 = spark.read.format("parquet").load("0917_2017_18_with_2017_cost.parquet")
df_og = df_2018.toPandas()

import importlib
import model_pipeline
importlib.reload(model_pipeline)
import pandas as pd
import numpy as np

import model_IAI
importlib.reload(model_IAI)
BIN_FLAG_COLUMNS = model_pipeline.get_bin_flag_columns(df_og) +['lab_monitoring_adherent','nephrology_consult_adherent','early_nephrology_referral']
STAGE_COLUMNS = [col for col in df_og.columns if "stage" in col.lower()]
#["stage_2017",'2017Q1_max_ckd_stage','2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage','2017Q4_max_ckd_stage']
CAT_COLUMNS = df_og.select_dtypes(include=["object","category"]).columns.tolist()
TRUE_NUM_COLUMNS = model_pipeline.get_true_num_columns(df_og,CAT_COLUMNS)+[ 'util_2017', 'total_increasing_quarters_2017'
, 'total_lab_tests', 'ckd_visit_count', 'quarters_with_labs', 'nephrology_visit_count', 'days_to_nephrology','MEDIAN_INCOME']
COST_COLUMNS = [col for col in df_og.columns if 
            "cost" in col.lower() or 
             "quarterly" in col.lower()  or "increasing" in col.lower()
             ]
UTILIZATION_COLUMNS = [col for col in df_og.columns if "claims" not in col.lower() ] + ['util_2017']
print("categorical cols: ", CAT_COLUMNS)
print("stage cols: ", STAGE_COLUMNS)
print(COST_COLUMNS)
leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, df_og.shape)  # preview first 50

def make_cost_stratum_3class(df):
    # Default to low-cost (class 0)
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df['highcost_gt_50000'] == 1) & (df['highcost_gt_100000'] == 0)] = 1
    # Emergent high cost (class 2): 100k to 200k
    cost_stratum[(df['highcost_gt_100000'] == 1) & (df['highcost_gt_200000'] == 0)] = 2
    # High cost (class 2): 200k+
    cost_stratum[df['highcost_gt_200000'] == 1] = 3
    return cost_stratum

# Add the new column to your data
df_og['cost_stratum_2018'] = make_cost_stratum_3class(df_og)
print(df_og["cost_stratum_2018"].value_counts(dropna=False))

cutoff_columns = [col for col in df_og.columns if col.startswith('highcost_gt_')]

feature_cols = [c for c in df_og.columns
              if c not in  (['annual_cost_2017','annual_cost_2018_deflated',"ENROLID", "cost_stratum_2018"] 
              + cutoff_columns)]    # keep only predictors excl. cost of 2018 and cutoff of 2017
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
# Columns to drop
high_corr_cols = corrs[corrs > 0.5].index.tolist()
# Remove the target column itself, if present
high_corr_cols = [col for col in high_corr_cols if col != "cost_stratum_2018"]
# Final filtered feature set
feature_cols = [col for col in feature_cols if col not in high_corr_cols]
print("High corr features dropped from prediction columns: ",high_corr_cols)
target_col = "highcost_gt_200000"

categorical cols:  ['ENROLID', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
stage cols:  ['stage_2017', '2017Q1_max_ckd_stage', '2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage', '2017Q4_max_ckd_stage']
['annual_cost_2017', 'highcost_gt_50000_2017', 'highcost_gt_75000_2017', 'highcost_gt_100000_2017', 'highcost_gt_200000_2017', 'highcost_gt_300000_2017', 'highcost_gt_400000_2017', 'highcost_gt_500000_2017', 'annual_cost_2018_deflated', 'highcost_gt_50000', 'highcost_gt_75000', 'highcost_gt_100000', 'highcost_gt_200000', 'highcost_gt_300000', 'highcost_gt_400000', 'highcost_gt_500000', '2017Q1_ckd_cost', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4

In [10]:

train_ids, test_ids,train_pd,test_pd = model_pipeline.train_test_split_enrol(df_og,
    target_col = "cost_stratum_2018",test_size=0.3,verbose=False,random_state=123)

print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = model_pipeline.train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")


Train shape: (23479, 96), Test shape: (10063, 96)
Feature cols: 78
Train: (23479, 96), Val: (5031, 96), Test: (5032, 96)
Train target distribution:
highcost_gt_200000
0    22819
1      660
Name: count, dtype: int64


In [ ]:
# DIAGNOSTIC: Check for object dtype columns in feature_cols
print("="*80)
print("DIAGNOSTIC: Checking for problematic columns")
print("="*80)

print(f"\nTotal feature_cols: {len(feature_cols)}")
print(f"CAT_COLUMNS in feature_cols: {len([c for c in CAT_COLUMNS if c in feature_cols])}")
print(f"TRUE_NUM_COLUMNS in feature_cols: {len([c for c in TRUE_NUM_COLUMNS if c in feature_cols])}")

# Check dtypes of all feature columns
dtypes_summary = train_pd[feature_cols].dtypes.value_counts()
print(f"\nDtype distribution in feature_cols:")
for dtype, count in dtypes_summary.items():
    print(f"  {dtype}: {count} columns")

# Find object columns
object_cols = train_pd[feature_cols].select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"\n⚠ Found {len(object_cols)} object-type columns in feature_cols:")
    for col in object_cols[:20]:  # Show first 20
        unique_vals = train_pd[col].nunique()
        sample_val = train_pd[col].iloc[0]
        print(f"  - {col}: {unique_vals} unique values, sample: {sample_val}")
    
    # Check if they're in CAT_COLUMNS or TRUE_NUM_COLUMNS
    print(f"\n  Checking categorization:")
    for col in object_cols:
        in_cat = col in CAT_COLUMNS
        in_num = col in TRUE_NUM_COLUMNS
        print(f"    {col}: CAT={in_cat}, NUM={in_num}")
else:
    print(f"\n✓ No object-type columns found in feature_cols")

# Check for columns not in either list
accounted = set([c for c in CAT_COLUMNS if c in feature_cols] + 
                [c for c in TRUE_NUM_COLUMNS if c in feature_cols])
unaccounted = set(feature_cols) - accounted

if unaccounted:
    print(f"\n⚠ {len(unaccounted)} feature_cols not in CAT_COLUMNS or TRUE_NUM_COLUMNS:")
    for col in list(unaccounted)[:20]:
        dtype = train_pd[col].dtype
        print(f"  - {col}: {dtype}")
else:
    print(f"\n✓ All feature_cols are in CAT_COLUMNS or TRUE_NUM_COLUMNS")

print("\n" + "="*80)

# Check ENROLID dtype (this might be the actual issue!)
print("\n🔍 Checking ENROLID column dtype:")
print(f"  train_pd['ENROLID'].dtype: {train_pd['ENROLID'].dtype}")
if train_pd['ENROLID'].dtype == 'object':
    print("  ⚠ ENROLID is object dtype - this will cause HDF5 error!")
    print("  Sample values:", train_pd['ENROLID'].head(3).tolist())
else:
    print("  ✓ ENROLID is numeric - should be fine for HDF5")

print("="*80)


DIAGNOSTIC: Checking for problematic columns

Total feature_cols: 78
CAT_COLUMNS in feature_cols: 7
TRUE_NUM_COLUMNS in feature_cols: 37

Dtype distribution in feature_cols:
  int32: 35 columns
  float64: 29 columns
  object: 7 columns
  int64: 7 columns

⚠ Found 7 object-type columns in feature_cols:
  - INCOME_LEVEL: 4 unique values, sample: 0
  - AGEGRP: 5 unique values, sample: 5
  - SEX: 2 unique values, sample: 2
  - REGION: 5 unique values, sample: 3
  - cost_pattern_2017: 3 unique values, sample: late_heavy
  - cost_stability_2017: 3 unique values, sample: moderate
  - lab_monitoring_intensity: 4 unique values, sample: High

  Checking categorization:
    INCOME_LEVEL: CAT=True, NUM=False
    AGEGRP: CAT=True, NUM=False
    SEX: CAT=True, NUM=False
    REGION: CAT=True, NUM=False
    cost_pattern_2017: CAT=True, NUM=False
    cost_stability_2017: CAT=True, NUM=False
    lab_monitoring_intensity: CAT=True, NUM=False

⚠ 34 feature_cols not in CAT_COLUMNS or TRUE_NUM_COLUMNS:
  - 

In [ ]:
# PRECOMPUTE ALL PAIRWISE DISTANCES USING precompute_distances.py
import precompute_distances
importlib.reload(precompute_distances)
from precompute_distances import (
    compute_distances_batched,
    save_distances_hdf5,
    save_distances_numpy_memmap,
    get_preprocessor
)
import h5py
import time
import os
import json

print("="*80)
print("PRECOMPUTING PAIRWISE DISTANCES")
print("="*80)

# 1. Separate majority and minority classes
print("\n1. Separating data by class...")
majority_df = train_pd[train_pd[target_col] == 0].copy()
minority_df = train_pd[train_pd[target_col] == 1].copy()

print(f"  Majority (class 0): {len(majority_df):,} samples")
print(f"  Minority (class 1): {len(minority_df):,} samples")
print(f"  Total distances to compute: {len(majority_df) * len(minority_df):,}")

# Store ENROLIDs and ensure they're numeric for HDF5
print("  Converting ENROLIDs to numeric format...")
if majority_df['ENROLID'].dtype == 'object':
    # Convert object/string ENROLIDs to int64
    majority_enrolids = majority_df['ENROLID'].astype('int64').values
    minority_enrolids = minority_df['ENROLID'].astype('int64').values
    print(f"    ✓ Converted object ENROLIDs to int64")
else:
    majority_enrolids = majority_df['ENROLID'].values
    minority_enrolids = minority_df['ENROLID'].values
    print(f"    ✓ ENROLIDs already numeric: {majority_enrolids.dtype}")

# 2. Preprocess features using PushPullSampler logic
print("\n2. Preprocessing features (matching PushPullSampler)...")

# Import PushPull preprocessing functions
import sys
sys.path.append('balancing_functions')
from model_pipeline import get_preprocessor, get_bin_flag_columns
from sklearn.impute import SimpleImputer

# Define exclusions (matching PushPullSampler)
exclude_cols = ["cost_stratum_2018"] + COST_COLUMNS + ["leaf_assignment", "predicted_cost_stratum"]
drop_cols = ['ENROLID', target_col] + exclude_cols
all_cols = [c for c in train_pd.columns if c not in drop_cols]

print(f"  Starting with {len(all_cols)} features (after excluding {len(drop_cols)} columns)")
print(f"    Excluded: ENROLID, {target_col}, cost columns, etc.")

# Combine majority and minority for consistent preprocessing
combined_df = pd.concat([
    majority_df[all_cols], 
    minority_df[all_cols]
], ignore_index=True)

# Categorize features (matching PushPullSampler logic)
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = combined_df.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"\n  Initial categorization:")
print(f"    Numeric: {len(numeric_cols)}")
print(f"    Categorical: {len(categorical_cols)}")

# Handle missing values (matching PushPullSampler)
if numeric_cols:
    imputer = SimpleImputer(strategy='median')
    combined_df[numeric_cols] = imputer.fit_transform(combined_df[numeric_cols])
    print(f"    ✓ Imputed numeric features (median)")

if categorical_cols:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    combined_df[categorical_cols] = cat_imputer.fit_transform(combined_df[categorical_cols])
    print(f"    ✓ Imputed categorical features (most_frequent)")

# Separate binary flags from numeric features
bin_feats = get_bin_flag_columns(combined_df)
num_feats = [c for c in numeric_cols if c not in bin_feats]

print(f"\n  Feature breakdown (matching PushPullSampler):")
print(f"    Categorical (will be one-hot encoded): {len(categorical_cols)}")
print(f"    Numeric (will be standardized): {len(num_feats)}")
print(f"    Binary flags (kept as-is): {len(bin_feats)}")
print(f"    Total: {len(categorical_cols) + len(num_feats) + len(bin_feats)}")

# Build preprocessor (matching PushPullSampler)
preprocessor = get_preprocessor(
    df=combined_df,
    categorical_cols=categorical_cols,
    numeric_cols=num_feats,
    verbose=False
)
# Transform
X_combined = preprocessor.fit_transform(combined_df)

# Split back into majority and minority
n_majority = len(majority_df)
X_majority = X_combined[:n_majority]
X_minority = X_combined[n_majority:]

print(f"  Preprocessed majority shape: {X_majority.shape}")
print(f"  Preprocessed minority shape: {X_minority.shape}")



PRECOMPUTING PAIRWISE DISTANCES

1. Separating data by class...
  Majority (class 0): 22,819 samples
  Minority (class 1): 660 samples
  Total distances to compute: 15,060,540
  Converting ENROLIDs to numeric format...
    ✓ Converted object ENROLIDs to int64

2. Preprocessing features (matching PushPullSampler)...
  Starting with 42 features (after excluding 57 columns)
    Excluded: ENROLID, highcost_gt_200000, cost columns, etc.

  Initial categorization:
    Numeric: 37
    Categorical: 5
    ✓ Imputed numeric features (median)
    ✓ Imputed categorical features (most_frequent)

  Feature breakdown (matching PushPullSampler):
    Categorical (will be one-hot encoded): 5
    Numeric (will be standardized): 12
    Binary flags (kept as-is): 25
    Total: 42
  Preprocessed majority shape: (22819, 52)
  Preprocessed minority shape: (660, 52)


In [15]:

# VALIDATION: Ensure no object dtypes remain
print("\n  Validating preprocessed data...")
if hasattr(X_majority, 'dtype'):
    # NumPy array
    if X_majority.dtype == 'object':
        raise TypeError(f"Preprocessed X_majority still has object dtype!")
    print(f"    ✓ X_majority dtype: {X_majority.dtype}")
    print(f"    ✓ X_minority dtype: {X_minority.dtype}")
else:
    # Might be sparse or other format
    try:
        X_majority_dense = X_majority.toarray() if hasattr(X_majority, 'toarray') else X_majority
        X_minority_dense = X_minority.toarray() if hasattr(X_minority, 'toarray') else X_minority
        print(f"    ✓ Converted to dense arrays")
        print(f"    ✓ X_majority dtype: {X_majority_dense.dtype}")
        print(f"    ✓ X_minority dtype: {X_minority_dense.dtype}")
        # Update references
        X_majority = X_majority_dense
        X_minority = X_minority_dense
    except Exception as e:
        print(f"    ⚠ Could not validate dtype: {e}")

# Additional check: ensure numeric
try:
    _ = X_majority.astype(np.float32)
    _ = X_minority.astype(np.float32)
    print(f"    ✓ Data is numeric (can be cast to float32)")
except (ValueError, TypeError) as e:
    print(f"    ✗ ERROR: Data contains non-numeric values!")
    print(f"    {e}")
    raise

# 3. Compute distances
print("\n3. Computing pairwise distances...")
print(f"  Shape: {X_majority.shape} (majority) × {X_minority.shape} (minority)")
print(f"  Total distances: {X_majority.shape[0] * X_minority.shape[0]:,}")

start_time = time.time()

# Check if tqdm is available
try:
    from tqdm import tqdm
    has_tqdm = True
    print("  ✓ Using tqdm for progress bar")
except ImportError:
    has_tqdm = False
    print("  ⚠ tqdm not available, no progress bar")

# Compute distances with explicit batching and progress
batch_size = 1000
n_majority = X_majority.shape[0]
n_minority = X_minority.shape[0]

# Pre-allocate
distances = np.zeros((n_majority, n_minority), dtype=np.float32)
print(f"\n  Allocated {distances.nbytes / 1e6:.1f} MB for distance matrix")

# Compute in batches
n_batches = (n_majority + batch_size - 1) // batch_size
print(f"  Computing in {n_batches} batches of {batch_size} samples...")

if has_tqdm:
    batch_iterator = tqdm(range(n_batches), desc="Computing distances")
else:
    batch_iterator = range(n_batches)
    print("  Progress: ", end="", flush=True)

for i in batch_iterator:
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_majority)
    
    # Compute distances for this batch
    from sklearn.metrics import pairwise_distances
    batch_distances = pairwise_distances(
        X_majority[start_idx:end_idx], 
        X_minority,
        metric='euclidean'
    )
    
    distances[start_idx:end_idx] = batch_distances.astype(np.float32)
    
    # Print progress every 10 batches if no tqdm
    if not has_tqdm and i % 10 == 0:
        print(f"{i}/{n_batches}", end=" ", flush=True)

if not has_tqdm:
    print()  # New line after progress

elapsed = time.time() - start_time
print(f"\n  ✓ Completed in {elapsed/60:.1f} minutes ({elapsed:.1f} seconds)")
print(f"  ✓ Distance range: [{distances.min():.3f}, {distances.max():.3f}]")
print(f"  ✓ Distance mean: {distances.mean():.3f}, std: {distances.std():.3f}")
print(f"  ✓ Matrix shape: {distances.shape}")
print(f"  ✓ Matrix size: {distances.nbytes / 1e6:.1f} MB")

# 4. Save in multiple formats
output_dir = "./precomputed_distances"
os.makedirs(output_dir, exist_ok=True)

print(f"\n4. Saving results to {output_dir}/...")

# Save HDF5 (recommended)
print("\n  a) Saving HDF5 format...")
save_distances_hdf5(
    distances,
    majority_enrolids,
    minority_enrolids,
    f"{output_dir}/distances_majority_minority.h5",
    compression='gzip'
)

# Save Numpy memmap
print("\n  b) Saving Numpy format...")
save_distances_numpy_memmap(
    distances,
    majority_enrolids,
    minority_enrolids,
    f"{output_dir}/distances"
)

# Save summary statistics
print("\n  c) Saving summary statistics...")
summary = {
    'n_majority': int(len(majority_enrolids)),
    'n_minority': int(len(minority_enrolids)),
    'n_distances': int(len(majority_enrolids) * len(minority_enrolids)),
    'n_features': int(X_majority.shape[1]),
    'distance_min': float(distances.min()),
    'distance_max': float(distances.max()),
    'distance_mean': float(distances.mean()),
    'distance_std': float(distances.std()),
    'distance_median': float(np.median(distances)),
    'dtype': str(distances.dtype),
    'compute_time_minutes': elapsed / 60,
    'batch_size': 1000,
    'preprocessing': {
        'n_cat_features': len(categorical_cols),
        'n_num_features': len(num_feats),
        'n_bin_features': len(bin_feats),
        'total_features_input': len(all_cols),
        'total_features_output': int(X_majority.shape[1])
    }
}

with open(f"{output_dir}/distance_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"     ✓ Saved metadata")

# Display file sizes
print("\n" + "="*80)
print("PRECOMPUTATION COMPLETE!")
print("="*80)
print(f"\nFiles saved to: {output_dir}/")
print(f"  - distances_majority_minority.h5")
print(f"      Size: {os.path.getsize(f'{output_dir}/distances_majority_minority.h5') / 1e6:.1f} MB")
print(f"  - distances_distances.npy")
print(f"      Size: {os.path.getsize(f'{output_dir}/distances_distances.npy') / 1e6:.1f} MB")
print(f"  - distances_majority_ids.npy, distances_minority_ids.npy")
print(f"  - distance_summary.json")

print(f"\nTotal computation time: {elapsed/60:.1f} minutes")
print(f"Average time per distance: {elapsed / (len(majority_enrolids) * len(minority_enrolids)) * 1e6:.2f} microseconds")

print("\n✓ Ready to use! See Cell 4 for loading examples.")

print("\n" + "="*80)
print("IMPORTANT: Preprocessing Details")
print("="*80)
print("The distances were computed using PushPullSampler's preprocessing:")
print("  1. Excluded: ENROLID, target, cost columns")
print("  2. Imputed: median (numeric), most_frequent (categorical)")
print("  3. Categorized: categorical → one-hot, numeric → standardized, binary → unchanged")
print("  4. This matches get_preprocessed_control_case_features() exactly")
print("\n✓ These distances can be directly reused in PushPull optimization!")
print("="*80)



  Validating preprocessed data...
    ✓ X_majority dtype: float64
    ✓ X_minority dtype: float64
    ✓ Data is numeric (can be cast to float32)

3. Computing pairwise distances...
  Shape: (22819, 52) (majority) × (660, 52) (minority)
  Total distances: 15,060,540
  ✓ Using tqdm for progress bar

  Allocated 60.2 MB for distance matrix
  Computing in 23 batches of 1000 samples...


Computing distances: 100%|██████████| 23/23 [00:00<00:00, 382.53it/s]


  ✓ Completed in 0.0 minutes (0.1 seconds)
  ✓ Distance range: [1.303, 46.403]
  ✓ Distance mean: 8.403, std: 4.831
  ✓ Matrix shape: (22819, 660)
  ✓ Matrix size: 60.2 MB

4. Saving results to ./precomputed_distances/...

  a) Saving HDF5 format...

Saving to HDF5: ./precomputed_distances/distances_majority_minority.h5


  ✓ Saved 53.4 MB

  b) Saving Numpy format...

Saving to Numpy memmap: ./precomputed_distances/distances
  ✓ Saved 60.4 MB (3 files)

  c) Saving summary statistics...
     ✓ Saved metadata

PRECOMPUTATION COMPLETE!

Files saved to: ./precomputed_distances/
  - distances_majority_minority.h5
      Size: 53.4 MB
  - distances_distances.npy
      Size: 60.2 MB
  - distances_majority_ids.npy, distances_minority_ids.npy
  - distance_summary.json

Total computation time: 0.0 minutes
Average time per distance: 0.00 microseconds

✓ Ready to use! See Cell 4 for loading examples.

IMPORTANT: Preprocessing Details
The distances were computed using PushPullSampler's preprocessing:
  1. Excluded: ENROLID, target, cost columns
  2. Imputed: median (numeric), most_frequent (categorical)
  3. Categorized: categorical → one-hot, numeric → standardized, binary → unchanged
  4. This matches get_preprocessed_control_case_features() exactly

✓ These distances can be directly reused in PushPull optimizati

In [ ]:
# PRECOMPUTE MAJORITY-TO-MAJORITY DISTANCES
# This is needed for the "push" objective in PushPull sampling
# (maximizing diversity among selected controls)

# Setup (in case previous cell wasn't run)
import os
import time
import json
import h5py
import numpy as np

output_dir = "./precomputed_distances"
os.makedirs(output_dir, exist_ok=True)

# Check if tqdm is available
try:
    from tqdm import tqdm
    has_tqdm = True
except ImportError:
    has_tqdm = False

print("="*80)
print("PRECOMPUTING MAJORITY-TO-MAJORITY DISTANCES")
print("="*80)

print("\n1. Estimating computation size...")
n_majority = len(majority_df)
total_distances = n_majority * n_majority
unique_distances = n_majority * (n_majority - 1) // 2  # Upper triangle only

print(f"  Majority samples: {n_majority:,}")
print(f"  Total distances (full matrix): {total_distances:,}")
print(f"  Unique distances (upper triangle): {unique_distances:,}")
print(f"  Memory (float32): {total_distances * 4 / 1e9:.2f} GB")

# Time estimation based on previous computation
prev_time = 0.05  # seconds for 15M distances
prev_count = 15_060_540
estimated_time = (total_distances / prev_count) * prev_time
print(f"\n  ⏱ Estimated time: {estimated_time:.1f} seconds ({estimated_time/60:.2f} minutes)")

print("\n2. Computing pairwise distances...")
print(f"  Shape: {X_majority.shape} × {X_majority.shape}")
print(f"  Using batched computation for memory efficiency...")

start_time = time.time()

# Compute in batches to manage memory
batch_size = 1000
n_batches = (n_majority + batch_size - 1) // batch_size

# Pre-allocate distance matrix
distances_majority = np.zeros((n_majority, n_majority), dtype=np.float32)
print(f"\n  Allocated {distances_majority.nbytes / 1e9:.2f} GB for distance matrix")
print(f"  Computing in {n_batches} batches of {batch_size} samples...")

if has_tqdm:
    batch_iterator = tqdm(range(n_batches), desc="Computing majority distances")
else:
    batch_iterator = range(n_batches)
    print("  Progress: ", end="", flush=True)

for i in batch_iterator:
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_majority)
    
    # Compute distances for this batch
    from sklearn.metrics import pairwise_distances
    batch_distances = pairwise_distances(
        X_majority[start_idx:end_idx], 
        X_majority,
        metric='euclidean'
    )
    
    distances_majority[start_idx:end_idx] = batch_distances.astype(np.float32)
    
    # Print progress every 10 batches if no tqdm
    if not has_tqdm and i % 10 == 0:
        print(f"{i}/{n_batches}", end=" ", flush=True)

if not has_tqdm:
    print()  # New line after progress

elapsed = time.time() - start_time
print(f"\n  ✓ Completed in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print(f"  ✓ Distance range: [{distances_majority[np.triu_indices_from(distances_majority, k=1)].min():.3f}, {distances_majority.max():.3f}]")
print(f"  ✓ Distance mean: {distances_majority[np.triu_indices_from(distances_majority, k=1)].mean():.3f}")
print(f"  ✓ Matrix shape: {distances_majority.shape}")
print(f"  ✓ Matrix size: {distances_majority.nbytes / 1e9:.2f} GB")

# Verify symmetry
is_symmetric = np.allclose(distances_majority, distances_majority.T, rtol=1e-5)
print(f"  ✓ Matrix is symmetric: {is_symmetric}")

# Verify diagonal is zero
diagonal_max = np.abs(np.diag(distances_majority)).max()
print(f"  ✓ Diagonal max: {diagonal_max:.6f} (should be ~0)")

print("\n3. Saving majority-to-majority distances...")

# Save HDF5
print("\n  a) Saving HDF5 format...")
h5_path = f"{output_dir}/distances_majority_majority.h5"
print(f"Saving to HDF5: {h5_path}")
with h5py.File(h5_path, 'w') as f:
    f.create_dataset('distances', data=distances_majority, compression='gzip', compression_opts=4)
    f.create_dataset('majority_enrolids', data=majority_enrolids)
    f.attrs['n_samples'] = n_majority
    f.attrs['n_distances'] = total_distances
    f.attrs['computation_time_seconds'] = elapsed
    f.attrs['matrix_type'] = 'symmetric'

file_size_mb = os.path.getsize(h5_path) / 1e6
print(f"  ✓ Saved {file_size_mb:.1f} MB")

# Save Numpy memmap
print("\n  b) Saving Numpy memmap format...")
npy_path = f"{output_dir}/distances_majority_majority"
np.save(f"{npy_path}_matrix.npy", distances_majority)
np.save(f"{npy_path}_enrolids.npy", majority_enrolids)

npy_size_mb = os.path.getsize(f"{npy_path}_matrix.npy") / 1e6
print(f"  ✓ Saved {npy_size_mb:.1f} MB")

# Update summary statistics
print("\n  c) Updating summary statistics...")
# Load existing summary
summary_path = f"{output_dir}/distance_summary.json"
with open(summary_path, 'r') as f:
    summary = json.load(f)

# Add majority-majority info
upper_tri_indices = np.triu_indices_from(distances_majority, k=1)
summary['majority_to_majority'] = {
    'n_samples': int(n_majority),
    'n_distances_total': int(total_distances),
    'n_distances_unique': int(unique_distances),
    'distance_min': float(distances_majority[upper_tri_indices].min()),
    'distance_max': float(distances_majority.max()),
    'distance_mean': float(distances_majority[upper_tri_indices].mean()),
    'distance_std': float(distances_majority[upper_tri_indices].std()),
    'distance_median': float(np.median(distances_majority[upper_tri_indices])),
    'compute_time_seconds': float(elapsed),
    'matrix_symmetric': bool(is_symmetric),
    'dtype': str(distances_majority.dtype)
}

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"     ✓ Updated {summary_path}")

print("\n" + "="*80)
print("MAJORITY-TO-MAJORITY DISTANCES COMPLETE!")
print("="*80)
print(f"\nFiles saved:")
print(f"  - {h5_path}")
print(f"      Size: {file_size_mb:.1f} MB (compressed)")
print(f"  - {npy_path}_matrix.npy")
print(f"      Size: {npy_size_mb:.1f} MB (uncompressed)")
print(f"\nComputation time: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print(f"Average time per distance: {elapsed / total_distances * 1e6:.2f} microseconds")


PRECOMPUTING MAJORITY-TO-MAJORITY DISTANCES

1. Estimating computation size...
  Majority samples: 22,819
  Total distances (full matrix): 520,706,761
  Unique distances (upper triangle): 260,341,971
  Memory (float32): 2.08 GB

  ⏱ Estimated time: 1.7 seconds (0.03 minutes)

2. Computing pairwise distances...
  Shape: (22819, 52) × (22819, 52)
  Using batched computation for memory efficiency...

  Allocated 2.08 GB for distance matrix
  Computing in 23 batches of 1000 samples...


Computing majority distances: 100%|██████████| 23/23 [00:01<00:00, 21.04it/s]



  ✓ Completed in 1.2 seconds (0.02 minutes)
  ✓ Distance range: [0.029, 48.990]
  ✓ Distance mean: 5.478
  ✓ Matrix shape: (22819, 22819)
  ✓ Matrix size: 2.08 GB
  ✓ Matrix is symmetric: True
  ✓ Diagonal max: 0.000001 (should be ~0)

3. Saving majority-to-majority distances...

  a) Saving HDF5 format...
Saving to HDF5: ./precomputed_distances/distances_majority_majority.h5
  ✓ Saved 1787.9 MB

  b) Saving Numpy memmap format...
  ✓ Saved 2082.8 MB

  c) Updating summary statistics...
     ✓ Updated ./precomputed_distances/distance_summary.json

MAJORITY-TO-MAJORITY DISTANCES COMPLETE!

Files saved:
  - ./precomputed_distances/distances_majority_majority.h5
      Size: 1787.9 MB (compressed)
  - ./precomputed_distances/distances_majority_majority_matrix.npy
      Size: 2082.8 MB (uncompressed)

Computation time: 1.2 seconds (0.02 minutes)
Average time per distance: 0.00 microseconds

USAGE NOTES

These majority-to-majority distances are used for the "PUSH" objective:
  - Maximizing 

In [11]:
# DIAGNOSTIC: Check if distances were actually computed
print("="*80)
print("DIAGNOSTIC: Checking computed distances")
print("="*80)

# Check if 'distances' variable exists
try:
    distances_exists = 'distances' in locals() or 'distances' in globals()
    if distances_exists:
        print("✓ distances variable exists")
        
        # Check shape
        print(f"\nDistance matrix shape: {distances.shape}")
        expected_shape = (len(majority_enrolids), len(minority_enrolids))
        print(f"Expected shape: {expected_shape}")
        
        if distances.shape == expected_shape:
            print("✓ Shape matches expected!")
        else:
            print(f"✗ Shape MISMATCH! Got {distances.shape}, expected {expected_shape}")
        
        # Check size
        n_distances = distances.shape[0] * distances.shape[1]
        print(f"\nTotal distances: {n_distances:,}")
        print(f"Expected: {len(majority_enrolids) * len(minority_enrolids):,}")
        
        # Check memory size
        memory_mb = distances.nbytes / 1e6
        print(f"\nMemory size: {memory_mb:.1f} MB")
        print(f"Data type: {distances.dtype}")
        
        # Expected size for float32: 4 bytes per number
        expected_mb = (distances.shape[0] * distances.shape[1] * 4) / 1e6
        print(f"Expected size (float32): {expected_mb:.1f} MB")
        
        # Check if distances are actually computed (not all zeros)
        print(f"\nDistance statistics:")
        print(f"  Min: {distances.min():.3f}")
        print(f"  Max: {distances.max():.3f}")
        print(f"  Mean: {distances.mean():.3f}")
        print(f"  Std: {distances.std():.3f}")
        
        if distances.min() == 0 and distances.max() == 0:
            print("  ⚠ WARNING: All distances are zero! This is wrong.")
        else:
            print("  ✓ Distances have reasonable values")
        
        # Sample a few distances
        print(f"\nSample distances (first 5x5 block):")
        print(distances[:5, :5])
        
    else:
        print("✗ distances variable does NOT exist!")
        print("  The computation may have failed silently.")
        
except Exception as e:
    print(f"✗ Error checking distances: {e}")

print("\n" + "="*80)


DIAGNOSTIC: Checking computed distances
✓ distances variable exists

Distance matrix shape: (22819, 660)
Expected shape: (22819, 660)
✓ Shape matches expected!

Total distances: 15,060,540
Expected: 15,060,540

Memory size: 60.2 MB
Data type: float32
Expected size (float32): 60.2 MB

Distance statistics:
  Min: 1.299
  Max: 47.303
  Mean: 8.313
  Std: 4.655
  ✓ Distances have reasonable values

Sample distances (first 5x5 block):
[[5.5592484 6.451542  6.029453  5.841557  5.2810574]
 [5.239002  6.609625  5.29605   5.603808  5.0151763]
 [5.1038013 6.458084  6.341217  5.080079  5.144444 ]
 [5.0147967 5.801651  5.8959026 5.3509426 6.086439 ]
 [4.382261  5.8613286 5.5855627 5.3545117 5.218709 ]]



In [12]:
# HOW TO LOAD AND USE PRECOMPUTED DISTANCES
# Run this cell after distances are precomputed

print("="*80)
print("LOADING PRECOMPUTED DISTANCES - EXAMPLES")
print("="*80)

output_dir = "./precomputed_distances"

# ============================================================================
# METHOD 1: Load from HDF5 (RECOMMENDED)
# ============================================================================
print("\n1. HDF5 Loading (memory-mapped, fast random access)")
print("-"*80)

import h5py

# Option A: Memory-mapped (doesn't load full array into RAM)
print("\n  a) Memory-mapped access (lazy loading):")
f = h5py.File(f"{output_dir}/distances_majority_minority.h5", 'r')
distances_hdf5 = f['distances']  # This is a dataset, not loaded yet
majority_ids_hdf5 = f['majority_enrolids'][:]
minority_ids_hdf5 = f['minority_enrolids'][:]

print(f"     Shape: {distances_hdf5.shape}")
print(f"     Dtype: {distances_hdf5.dtype}")
print(f"     Memory used: ~0 MB (memory-mapped)")

# Example 1: Get distances for one minority case
case_enrolid = minority_ids_hdf5[0]  # First minority case
case_idx = 0
case_distances = distances_hdf5[:, case_idx]  # Gets one column efficiently
print(f"\n     Example: Distances for minority case {case_enrolid}")
print(f"       Got {len(case_distances):,} distances")
print(f"       Min distance: {case_distances.min():.3f}")
print(f"       Top-5 nearest majority indices: {np.argsort(case_distances)[:5]}")

# Example 2: Get top-K nearest controls for a minority case
k = 100
case_idx = 5
nearest_k_indices = np.argpartition(distances_hdf5[:, case_idx], k)[:k]
nearest_k_distances = distances_hdf5[nearest_k_indices, case_idx]
sorted_order = np.argsort(nearest_k_distances)
nearest_k_indices = nearest_k_indices[sorted_order]

print(f"\n     Example: Top-{k} nearest controls for minority case {minority_ids_hdf5[case_idx]}")
print(f"       Nearest control ENROLID: {majority_ids_hdf5[nearest_k_indices[0]]}")
print(f"       Distance: {nearest_k_distances[sorted_order[0]]:.3f}")

# Don't forget to close when done
f.close()

# Option B: Load full array into memory (if you have enough RAM)
print("\n  b) Load full array into RAM:")
with h5py.File(f"{output_dir}/distances_majority_minority.h5", 'r') as f:
    distances_full = f['distances'][:]  # [:] loads entire array
    majority_ids = f['majority_enrolids'][:]
    minority_ids = f['minority_enrolids'][:]

print(f"     Loaded {distances_full.nbytes / 1e6:.1f} MB into RAM")

# ============================================================================
# METHOD 2: Load from Numpy (FASTEST, but uses more disk space)
# ============================================================================
print("\n\n2. Numpy memmap loading (fastest access)")
print("-"*80)

# Load as memory-mapped array (doesn't load into RAM)
distances_mmap = np.load(f"{output_dir}/distances_matrix.npy", mmap_mode='r')
majority_ids_npy = np.load(f"{output_dir}/majority_enrolids.npy")
minority_ids_npy = np.load(f"{output_dir}/minority_enrolids.npy")

print(f"  Shape: {distances_mmap.shape}")
print(f"  Memory used: ~0 MB (memory-mapped)")

# Access is exactly like a numpy array
print(f"\n  Example: Random access test")
for i in range(3):
    case_idx = np.random.randint(0, len(minority_ids_npy))
    ctrl_idx = np.random.randint(0, len(majority_ids_npy))
    dist = distances_mmap[ctrl_idx, case_idx]
    print(f"    Distance from {majority_ids_npy[ctrl_idx]} to {minority_ids_npy[case_idx]}: {dist:.3f}")

# ============================================================================
# METHOD 3: Helper Functions for Common Queries
# ============================================================================
print("\n\n3. Helper functions for common queries")
print("-"*80)

def get_topk_nearest_controls(distances, majority_ids, minority_ids, case_enrolid, k=10):
    """Get k nearest majority controls for a given minority case"""
    case_idx = np.where(minority_ids == case_enrolid)[0][0]
    case_distances = distances[:, case_idx]
    
    # Get k-nearest using argpartition (faster than full sort)
    nearest_k_indices = np.argpartition(case_distances, k)[:k]
    nearest_k_distances = case_distances[nearest_k_indices]
    
    # Sort the k nearest
    sorted_order = np.argsort(nearest_k_distances)
    nearest_k_indices = nearest_k_indices[sorted_order]
    nearest_k_distances = nearest_k_distances[sorted_order]
    
    return majority_ids[nearest_k_indices], nearest_k_distances

def get_controls_within_radius(distances, majority_ids, minority_ids, case_enrolid, radius=2.0):
    """Get all majority controls within distance radius of a minority case"""
    case_idx = np.where(minority_ids == case_enrolid)[0][0]
    case_distances = distances[:, case_idx]
    
    within_radius = case_distances <= radius
    return majority_ids[within_radius], case_distances[within_radius]

def create_knn_lookup(distances, majority_ids, minority_ids, k=100):
    """
    Precompute k-nearest neighbors for ALL minority cases
    Returns dict: {minority_enrolid: (nearest_majority_ids, distances)}
    """
    print(f"  Creating k={k} nearest neighbor lookup for {len(minority_ids)} cases...")
    knn_lookup = {}
    
    for case_idx in tqdm(range(len(minority_ids))):
        case_enrolid = minority_ids[case_idx]
        case_distances = distances[:, case_idx]
        
        nearest_k_indices = np.argpartition(case_distances, k)[:k]
        nearest_k_distances = case_distances[nearest_k_indices]
        sorted_order = np.argsort(nearest_k_distances)
        
        knn_lookup[case_enrolid] = (
            majority_ids[nearest_k_indices[sorted_order]],
            nearest_k_distances[sorted_order]
        )
    
    return knn_lookup

# Example usage
print("\n  Example 1: Top-10 nearest controls for a case")
case_id = minority_ids_npy[0]
nearest_ids, nearest_dists = get_topk_nearest_controls(
    distances_mmap, majority_ids_npy, minority_ids_npy, case_id, k=10
)
print(f"    Case {case_id} nearest controls: {nearest_ids[:3]} (distances: {nearest_dists[:3]})")

print("\n  Example 2: All controls within radius 3.0")
within_radius_ids, within_radius_dists = get_controls_within_radius(
    distances_mmap, majority_ids_npy, minority_ids_npy, case_id, radius=3.0
)
print(f"    Found {len(within_radius_ids)} controls within radius 3.0")

# ============================================================================
# METHOD 4: Integration with PushPull Sampler
# ============================================================================
print("\n\n4. Integration with PushPull sampler")
print("-"*80)
print("""
To use precomputed distances in your PushPull sampler, modify the sampler code:

Option A: Pass precomputed distances directly
  - Skip the distance computation step
  - Use the precomputed matrix for NN_case lookup
  - Still compute control-control distances (smaller)

Option B: Use as a filter/heuristic
  - Use precomputed distances to prune candidates before MILP
  - Example: Only consider controls within top-K or radius-R
  - Drastically reduces MILP problem size

Example modification in PushPullSampler:
  # Instead of:
  # NN_case, D_pn = self._topk_prune_case_to_control(...)
  
  # Use:
  # Load precomputed distances for this leaf
  # D_pn = precomputed_distances[control_indices][:, case_indices]
  # NN_case = get_topk_from_precomputed(D_pn, top_k_case_ctrl)
""")

print("\n" + "="*80)
print("LOADING EXAMPLES COMPLETE")
print("="*80)
print("\nRecommended workflow:")
print("1. Use HDF5 for general purpose (good compression, fast access)")
print("2. Use numpy memmap if you need maximum speed and have disk space")
print("3. Create KNN lookup dict if you repeatedly query same cases")
print("4. Use helper functions for common query patterns")


LOADING PRECOMPUTED DISTANCES - EXAMPLES

1. HDF5 Loading (memory-mapped, fast random access)
--------------------------------------------------------------------------------

  a) Memory-mapped access (lazy loading):
     Shape: (22819, 660)
     Dtype: float32
     Memory used: ~0 MB (memory-mapped)

     Example: Distances for minority case 764339802
       Got 22,819 distances
       Min distance: 2.643
       Top-5 nearest majority indices: [4703 1829 2997 1141 1161]


TypeError: Indexing elements must be in increasing order